Ejecuta la siguiente celda e introduce la contraseña facilitada en el curso para desbloquear el cuaderno.


In [ ]:
%pip install -q git+https://github.com/dcrespons14/cam-estrategia-aula.git
from cam_estrategia import desbloquear_cuaderno
desbloquear_cuaderno()


# Aprendizaje por refuerzo: la estrategia de Piastri

En el cuaderno de la clase 4.6 construimos un **modelo de carrera** de México y vimos que Piastri, con el tercer mejor ritmo base, acaba casi siempre 5.º con la estrategia que usó en 2025.

En este cuaderno dejamos que Piastri **aprenda su propia estrategia** con **aprendizaje por refuerzo**: el resto de pilotos mantienen sus estrategias de 2025 y Piastri corre miles de carreras, prueba estrategias distintas y se queda con la que más puntos le da.

## Inicialización

Importamos las librerías relevantes.

In [ ]:
# Librerías propias (ver src/race_analysis y src/race_models)
from race_analysis.ingestion.openf1 import get, load_race
from race_analysis.ingestion.laps import clean_data
from race_analysis.tyre_deg.pace import fit_pooled_race_pace
from race_analysis.overtaking_model.overtaking import find_overtakes, attack_situations, fit_logistic
from race_models.f1.config import RaceConfig
from race_models.f1.models import base_pace_from_race_pace, tyre_deg_from_race_pace
from race_models.f1.engine.simulator import simulate_race
from race_models.f1.engine.strategy import strategy_name
from race_models.f1.analysis.race_plots import plot_race_trace, plot_tyre_strategies
from race_models.f1.rl.env import StrategyEnv, POINTS
from race_models.f1.rl.monte_carlo import MonteCarloAgent
from race_models.f1.rl.q_learning import QLearningAgent
from race_models.f1.analysis.rl_plots import plot_learning_curve, plot_q_values

# Librerías de Python
import numpy as np
import pandas as pd
from IPython.display import display

## 1. El modelo de carrera

Repetimos el modelo de carrera del cuaderno de la clase 4.6, con los mismos datos de México 2023-2025: ritmo base, modelo lineal de neumáticos y modelo de adelantamiento.

In [ ]:
TRACK = "Mexico"
YEARS = [2023, 2024, 2025]
FUEL_EFFECT = 0.05  # s por vuelta de combustible a bordo
MAX_GAP = 1.5       # distancia máxima (s) con el coche de delante para considerar que hay un ataque

races = {year: load_race(year, TRACK) for year in YEARS}
laps = {year: clean_data(race) for year, race in races.items()}
for frame in laps.values():
    frame["fuel_corrected"] = frame["lap_time"] - FUEL_EFFECT * frame["laps_remaining"]

# Modelo de neumáticos y ritmo base
linear = fit_pooled_race_pace(pd.concat(laps.values(), ignore_index=True), model="lineal")
pace = linear[2025]
tlas = {driver["driver_number"]: driver["name_acronym"] for driver in get("drivers", session_key=races[2025].session_key)}

# Modelo de adelantamiento
overtaking_data = {year: find_overtakes(race, laps[year], max_gap=MAX_GAP) for year, race in races.items()}
attacks = pd.concat([attack_situations(races[year], laps[year], linear[year], overtaking_data[year], max_gap=MAX_GAP)
                     for year in YEARS], ignore_index=True)
overtaking_model = fit_logistic(attacks["delta_pace"], attacks["overtake"])

config = RaceConfig(base_pace=base_pace_from_race_pace(pace, tlas), tyre_deg=tyre_deg_from_race_pace(pace),
                    overtaking_lr=[overtaking_model.intercept, overtaking_model.slope])
config

## 2. La estrategia real de Piastri

Usamos las estrategias de 2025 del cuaderno de la clase 4.6. Piastri hizo dos paradas: **S - M24 - S47**.

In [ ]:
strategies_2025 = {
    "NOR": [("S", 34), ("M", 37)],
    "LEC": [("S", 31), ("M", 40)],
    "HAM": [("S", 23), ("M", 24), ("S", 24)],
    "RUS": [("S", 25), ("M", 23), ("S", 23)],
    "VER": [("M", 37), ("S", 34)],
    "ANT": [("S", 22), ("M", 25), ("S", 24)],
    "PIA": [("S", 24), ("M", 23), ("S", 24)],
    "HAD": [("M", 34), ("S", 37)],
    "BEA": [("S", 24), ("M", 24), ("S", 23)],
    "TSU": [("M", 36), ("S", 35)],
    "OCO": [("S", 27), ("M", 44)],
    "SAI": [("M", 17), ("S", 29), ("S", 25)],
    "HUL": [("S", 25), ("M", 46)],
    "ALO": [("S", 20), ("M", 51)],
    "BOR": [("M", 37), ("S", 34)],
    "ALB": [("H", 41), ("S", 30)],
    "GAS": [("M", 33), ("S", 38)],
    "STR": [("S", 26), ("M", 17), ("S", 28)],
    "COL": [("H", 48), ("S", 23)],
}
RIVALS = ["PIA", "NOR", "VER", "LEC", "HAM", "RUS", "ANT", "BEA"]

Como en la clase 4.6, para valorar una estrategia corremos **muchas carreras** y miramos la distribución de resultados. Usamos siempre las mismas 200 carreras (semillas), distintas de las que usará Piastri para aprender.

In [ ]:
EVALUATION_SEEDS = range(100_000, 100_200)  # carreras de evaluación: el agente no las ve al entrenar


def evaluate(stints):
    """Posición final de Piastri en las 200 carreras de evaluación con una estrategia fija."""
    positions = []
    for seed in EVALUATION_SEEDS:
        race = simulate_race(config, {**strategies_2025, "PIA": stints}, seed=seed)
        positions.append(race.final_order.index("PIA") + 1)
    return pd.Series(positions)


def summary(positions):
    """Tabla con la posición y los puntos de cada estrategia, como en la clase 4.6."""
    positions = pd.DataFrame(positions)
    points = positions.apply(lambda column: column.map(POINTS).fillna(0))
    table = pd.DataFrame({
        "Posición media": positions.mean(),
        "Puntos medios": points.mean(),
        "Puntos STD": points.std(),
        "Podio": (positions <= 3).mean(),
        "Mejor": positions.min(),
        "Peor": positions.max(),
    })
    with pd.option_context("display.float_format", "{:.2f}".format):
        display(table.sort_values("Puntos medios", ascending=False))


real_positions = evaluate(strategies_2025["PIA"])
summary({"S - M24 - S47 (real)": real_positions})

## 3. La carrera como un problema de aprendizaje por refuerzo

En aprendizaje por refuerzo hay un **agente** que toma decisiones en un **entorno**, y recibe una **recompensa** según lo bien que lo haya hecho. Como un perro que aprende a base de galletas.

- **Agente**: el muro de Piastri.
- **Entorno**: el modelo de carrera, con los demás pilotos y sus estrategias fijas.
- **Acción**: un **stint**, `(compuesto, vueltas)`, igual que en las estrategias de arriba. El agente decide en la salida y en cada parada: `("H", 40)` es montar duros y parar a las 40 vueltas.
- **Estado**: lo que sabe el muro cuando decide: la vuelta, los compuestos que ya ha usado y las paradas que ha hecho. Las paradas son importantes: llegar a la vuelta 42 con una parada o con dos no es la misma carrera, porque cada parada cuesta unos 20 s. Sin ellas, el agente mezcla las dos situaciones y puede acabar creyendo que una estrategia de tres paradas es tan buena como una de una.
- **Recompensa**: los **puntos** de la posición final. Si no usa dos compuestos, como obliga el reglamento, la recompensa es **-25** (una descalificación).

Para no perder el tiempo con estrategias absurdas, cada stint tiene al menos 5 vueltas y hay como mucho 3 paradas.

¿Por qué decidimos stint a stint y no vuelta a vuelta ("¿paro en esta vuelta?")? Porque son 71 decisiones por carrera, y al principio el agente casi nunca para cuando debe: aprende que parar pronto le asegura los dos compuestos y se queda ahí. Con stints son de 1 a 4 decisiones por carrera, y aprende mucho más rápido.

In [ ]:
env = StrategyEnv(config, strategies_2025, driver="PIA")

state = env.reset(seed=0)
actions = env.legal_actions()
print("Estado en la salida:", state)
print(f"Acciones posibles: {len(actions)}, por ejemplo {actions[:3]} ... {actions[-3:]}")

Corremos una carrera con la estrategia real, stint a stint. La recompensa es 0 hasta la bandera a cuadros.

In [ ]:
state = env.reset(seed=0)
for stint in strategies_2025["PIA"]:
    state, reward, done = env.step(stint)
    print(f"Stint {stint} -> estado {state}, recompensa {reward}, fin de carrera: {done}")
print(f"{env.strategy_name()}: Piastri acaba {env.position()}.º")

## 4. Q-learning tabular

El agente aprende una **tabla** `Q(estado, acción)`: los **puntos que espera** si en ese estado elige ese stint y después sigue eligiendo bien. Al principio la tabla está vacía. Después de cada carrera, actualizamos todas las decisiones que se han tomado:

$$Q(s, a) \leftarrow Q(s, a) + \frac{\text{objetivo} - Q(s, a)}{N(s, a)}$$

$N(s, a)$ es el número de veces que se ha probado esa acción en ese estado, así que $Q$ es la **media** de los objetivos. El objetivo depende del stint:

- **Último stint**: los puntos de la carrera.
- **Los demás**: el mejor valor de la tabla en el estado al que lleva ese stint, $\max_{a'} Q(s', a')$. Es decir, cuántos puntos podemos sacar a partir de ahí, eligiendo bien el resto de la carrera.

### Exploración: UCB

Si el agente eligiera siempre el mejor stint de la tabla, se quedaría con el primero que le saliera bien. Tiene que **explorar**: probar también stints que parecen peores, por si lo son solo por mala suerte. Usamos **UCB** (*upper confidence bound*): primero prueba una vez cada stint posible y, a partir de ahí, elige el que tenga el mayor

$$Q(s, a) + c\,\sqrt{\frac{\ln N(s)}{N(s, a)}}$$

donde $N(s)$ es el número de veces que ha decidido en ese estado. El segundo término es un **bonus** para los stints que ha probado pocas veces: cuantas más carreras tiene un stint, menos bonus, y más se fía de su media. La constante $c$ (en puntos) dice cuánto explora; usamos $c = 10$.

Una alternativa muy habitual es **ε-greedy**: elegir un stint al azar con una probabilidad ε que baja de 1 a 0 durante el entrenamiento. Aquí funciona peor por dos motivos:

- Cuando ε llega a 0, los stints que salieron mal al principio no se vuelven a probar, y el agente se queda con la primera estrategia razonable que encontró.
- Como ε depende del número total de carreras, entrenar 20.000 o 30.000 carreras da entrenamientos distintos y, a veces, estrategias distintas. Con UCB no: entrenar 30.000 carreras es lo mismo que entrenar 20.000 y seguir 10.000 más.

### Monte Carlo

Existe una versión todavía más sencilla, **Monte Carlo**, en la que el objetivo de todas las decisiones son los puntos de la carrera. El problema es la exploración: si un buen primer stint va seguido de un segundo stint de exploración que sale mal, Monte Carlo culpa también al primer stint. Lo vemos con dos carreras:

In [ ]:
first = ("salida", ("H", 40))  # (estado, acción) de la primera decisión
for agent in [MonteCarloAgent(), QLearningAgent()]:
    agent.update([first, ("parada", ("S", 31))], 15.0)  # buena carrera: 3.º
    agent.update([first, ("parada", ("H", 31))], -25.0)  # segundo stint al azar: no usa dos compuestos
    print(f"{type(agent).__name__}: Q(salida, H 40 vueltas) = {agent.q[first]:+.1f} puntos")

Para Monte Carlo, empezar con duro 40 vueltas vale -5 puntos; para Q-learning, 15, porque sabe que después se puede poner el blando. En nuestra carrera, Monte Carlo necesita muchas más carreras para dar resultados fiables.

## 5. Entrenamiento

Piastri corre 30.000 carreras. Tarda unos minutos.

Todo el entrenamiento es **repetible**: la carrera número `i` usa la semilla `SEED * 1_000_000 + i`, y la semilla del agente decide el orden en el que prueba los stints la primera vez. Con la misma `SEED`, el resultado es exactamente el mismo.

Cada 1.000 carreras, el entrenamiento **comprueba** qué estrategia ha aprendido hasta ese momento: la corre sin explorar en 50 carreras fijas, que no usa para aprender. Cuando esa estrategia deja de cambiar, el entrenamiento ha **convergido**.

In [ ]:
SEED = 0            # cambiadla para repetir el entrenamiento con otras carreras
EPISODES = 30_000

agent = QLearningAgent(seed=SEED, exploration=10.0)
history = agent.train(env, episodes=EPISODES, seed=SEED * 1_000_000, progress=True)
history.tail()

La **curva de aprendizaje** tiene dos líneas:

- **Entrenamiento**: los puntos medios de las últimas 500 carreras, explorando. Al principio el agente prueba todos los stints y muchas veces no usa dos compuestos; poco a poco se concentra en los buenos. Nunca llega del todo al valor de la estrategia aprendida, porque UCB sigue probando otras de vez en cuando. Las caídas bruscas son esas rondas de exploración: el bonus de los stints olvidados crece poco a poco y, de vez en cuando, el agente vuelve a probarlos durante unas cuantas carreras.
- **Estrategia aprendida**: los puntos de la estrategia que elegiría el agente sin explorar, en cada comprobación. Cuando se queda plana, el agente ha convergido.

In [ ]:
plot_learning_curve(history, window=500, references={"S - M24 - S47 (real)": real_positions.map(POINTS).fillna(0).mean()},
                    title="Piastri · Curva de aprendizaje");

Estas son las veces que ha cambiado la estrategia aprendida durante el entrenamiento:

In [ ]:
checks = history.dropna(subset=["learned"])
changes = checks[checks["learned"] != checks["learned"].shift()]
changes[["episode", "learned", "learned_points"]].rename(
    columns={"episode": "Carrera", "learned": "Estrategia aprendida", "learned_points": "Puntos (50 carreras)"})

In [ ]:
print("Estrategias más usadas en las últimas 2.000 carreras de entrenamiento:")
history.tail(2_000)["strategy"].value_counts().head(5)

## 6. ¿Qué ha aprendido Piastri?

Con la tabla aprendida, el agente ya no explora: en cada decisión elige el stint con más puntos esperados.

In [ ]:
reward, decisions = agent.play(env, seed=EVALUATION_SEEDS[0])
learned_stints = list(env.stints)
learned_name = env.strategy_name()
print(f"Estrategia aprendida: {learned_name}  {learned_stints}")
print(f"En esta carrera, Piastri acaba {env.position()}.º ({reward:.0f} puntos)")

Así ve la tabla Q la primera decisión: los puntos esperados para cada compuesto según la longitud del primer stint. Solo dibujamos los stints probados al menos 5 veces, y cuanto más grande es el punto, más carreras tiene.

In [ ]:
plot_q_values(agent, decisions[0][0], min_visits=5, title="Piastri · Primer stint: puntos esperados");

- Hacer toda la carrera sin parar no aparece: vale -25 puntos (no usa dos compuestos), y el agente lo descarta en cuanto lo prueba.
- El stint elegido tiene muchas más carreras que el resto: UCB se concentra en lo que parece bueno. Los demás tienen menos carreras y más ruido, y algunos se probaron sobre todo al principio, cuando el agente todavía no sabía qué hacer después. La tabla sirve para **decidir**, pero para comparar con precisión dos estrategias parecidas hay que correrlas muchas veces (sección 8).

Y la decisión de la parada: qué neumático poner y cuántas vueltas hacer con él.

In [ ]:
if len(decisions) > 1:
    plot_q_values(agent, decisions[1][0], min_visits=5,
                  title=f"Piastri · Stint desde la vuelta {decisions[1][0].lap}: puntos esperados");

In [ ]:
race = env.result()
plot_race_trace(race, drivers=RIVALS, title=f"Piastri · {learned_name} (aprendida)");
plot_tyre_strategies(race, title=f"Piastri · {learned_name} (aprendida)");

## 7. ¿Es mejor que la estrategia real?

Comparamos las dos estrategias en las mismas 200 carreras de evaluación, que el agente no ha visto al entrenar.

In [ ]:
learned_positions = evaluate(learned_stints)
summary({"S - M24 - S47 (real)": real_positions, f"{learned_name} (aprendida)": learned_positions})

## 8. ¿Nos podemos fiar del resultado?

El agente ha aprendido con carreras aleatorias, así que conviene comprobar que no se ha quedado con una estrategia buena pero no la mejor. Con **una parada**, podemos probar **todas** las estrategias (fuerza bruta): 3 compuestos de salida, 2 para el segundo stint y unas 60 vueltas de parada, unas 370 estrategias. Corremos 20 carreras de cada una para quedarnos con las 10 mejores, y después esas 10 en 1.000 carreras más. Tarda unos minutos.

In [ ]:
SCREENING_SEEDS = range(200_000, 200_020)
TOTAL_LAPS = config.total_laps

one_stop = {}  # nombre -> stints
for first in "SMH":
    for second in "SMH":
        if first != second:
            for lap in range(env.min_stint, TOTAL_LAPS - env.min_stint + 1):
                stints = [(first, lap), (second, TOTAL_LAPS - lap)]
                one_stop[strategy_name(stints)] = stints

# env.run_strategy corre una carrera con una estrategia fija y devuelve la recompensa (puntos).
screening = pd.Series({name: np.mean([env.run_strategy(stints, seed) for seed in SCREENING_SEEDS])
                       for name, stints in one_stop.items()})
best = screening.sort_values(ascending=False).head(10)
print(f"{len(one_stop)} estrategias de una parada. Las 10 mejores en las 20 carreras de criba (puntos medios):")
best.round(2)

Las diferencias entre las mejores son de décimas de punto, y con 200 carreras el ruido es de unos ±0,3 puntos. Para separarlas usamos **1.000 carreras, las mismas para todas las estrategias**: así la suerte (adelantamientos, errores…) es igual para todas y la diferencia entre dos estrategias es mucho más precisa que cada media por separado.

In [ ]:
FINAL_SEEDS = range(400_000, 401_000)

candidates = {name: one_stop[name] for name in best.index}
candidates.setdefault(learned_name, learned_stints)
final = pd.DataFrame({name: [env.run_strategy(stints, seed) for seed in FINAL_SEEDS]
                      for name, stints in candidates.items()})
winner = final.mean().idxmax()
difference = final.sub(final[winner], axis=0)
table = pd.DataFrame({
    "Puntos medios": final.mean(),
    "Diferencia con la mejor": difference.mean(),
    "± (error de la diferencia)": difference.std() / np.sqrt(len(FINAL_SEEDS)),
}).sort_values("Puntos medios", ascending=False)
table.index = [f"{name} (aprendida)" if name == learned_name else name for name in table.index]
table.round(2)

La mejor estrategia de una parada es salir con **duro y poner blando hacia la vuelta 42**, pero entre parar en la vuelta 40 y en la 44 hay menos de 0,2 puntos: es una **ventana**, no una vuelta exacta. El agente ha aprendido una estrategia de esa ventana (o muy cerca), con unos 4 puntos más por carrera que la estrategia real.

Distinguir dentro de la ventana necesitaría muchas más carreras de entrenamiento: con el ruido de una carrera, separar dos estrategias que se diferencian en 0,1 puntos exige miles de carreras de cada una. En la práctica, un estratega no necesita más precisión que esa: cualquier vuelta de la ventana es buena, y la decisión final la darán el tráfico, el Safety Car o lo que hagan los rivales.

Y ha llegado ahí **sin probarlas todas**: también ha explorado estrategias de dos y tres paradas, donde la fuerza bruta ya serían decenas de miles de estrategias. Es la idea habitual: el aprendizaje por refuerzo encuentra rápido la zona buena, y después se comparan unas pocas candidatas con muchas simulaciones.

## 9. Limitaciones y siguientes pasos

- **Rivales fijos**: los demás no reaccionan a lo que hace Piastri. En la realidad, si Piastri para antes, McLaren o Mercedes podrían cubrirle. Para eso hacen falta varios agentes que aprenden a la vez (*multi-agent learning*).
- **Estrategia fija**: con este estado, el agente no sabe en qué posición va, así que aprende una estrategia fija. Con `StrategyEnv(..., position_in_state=True)` el estado incluye la posición al salir de boxes, y el agente puede elegir el siguiente stint según cómo vaya la carrera. Necesita más carreras para aprender, porque hay más estados.
- **Tabla**: con estados más ricos (huecos, Safety Car, degradación real…), la tabla crece demasiado y hay que sustituirla por una función, por ejemplo una red neuronal (*deep reinforcement learning*).

**Ejercicios**

1. Entrenad un `MonteCarloAgent` con las mismas carreras y comparad la estrategia que aprende con la de Q-learning.
2. Repetid el entrenamiento con `SEED = 1` y `SEED = 2`. ¿Aprende la misma estrategia? ¿Está dentro de la ventana de la sección 8?
3. Cambiad la exploración: `QLearningAgent(seed=SEED, exploration=3.0)`. ¿Qué pasa si el agente explora poco?
4. Haced que aprenda otro piloto, por ejemplo Leclerc, y comparad con las opciones de la clase 4.6.
5. Probad `position_in_state=True` con 60.000 carreras. ¿Cambia la estrategia según la posición?